# Hunyuan3D-2.1 테스트 (Kaggle)
단일 이미지 → 3D 모델 변환

## 실행 전 필수 설정
1. 오른쪽 상단 **Session options** → **Accelerator**: `GPU T4 x2` 또는 `GPU P100`
2. 오른쪽 상단 **Session options** → **Internet**: `On`
3. 오른쪽 패널 **Add Data** → 이미지 파일을 데이터셋으로 추가
   - 추가한 데이터셋은 `/kaggle/input/<dataset-slug>/` 경로에 마운트됩니다

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. numpy 버전 확인 (충돌 시 아래 주석 해제 후 실행 → 커널 재시작)
import numpy as np
print(f'numpy 버전: {np.__version__}')
# 버전 충돌이 발생하면 아래 줄 주석 해제:
# !pip install numpy==1.26.4 -q && echo '완료 - 커널을 재시작하세요'

In [ ]:
# 3. 저장소 클론 및 의존성 설치
import os, glob, sys

REPO_DIR = '/kaggle/working/Hunyuan3D-2.1'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('저장소 이미 존재, 클론 건너뜀')

%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q

# custom_rasterizer: build_ext --inplace 로 CUDA .so를 소스 디렉토리에 생성
!pip install ninja -q
_rast_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'rasterizer' in d.lower() and os.path.exists(f'{d}/setup.py')),
    None
)
if _rast_dir:
    print(f'custom_rasterizer 빌드 중: {_rast_dir}')
    # install 대신 build_ext --inplace: .so 파일이 소스 디렉토리에 직접 생성됨
    !cd {_rast_dir} && python setup.py build_ext --inplace
    print('custom_rasterizer 빌드 완료')
else:
    print('custom_rasterizer setup.py 없음')

print('설치 완료')

In [ ]:
# 4. Kaggle 입력 데이터셋에서 이미지 탐색
import glob

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp',
                    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if image_files:
    image_path = image_files[0]
    print(f'입력 이미지: {image_path}')
    if len(image_files) > 1:
        print(f'  (총 {len(image_files)}개 발견, 첫 번째 사용)')
        for f in image_files:
            print(f'    {f}')
else:
    print('이미지를 찾을 수 없습니다.')
    print('Kaggle 노트북 오른쪽 패널 > Add Data 에서 이미지 데이터셋을 추가하세요.')
    raise FileNotFoundError('입력 이미지 없음')

In [ ]:
# 5. 배경 제거
import sys
sys.path.insert(0, '/kaggle/working/Hunyuan3D-2.1/hy3dshape')

import matplotlib.pyplot as plt
from PIL import Image
from hy3dshape.rembg import BackgroundRemover

input_img = Image.open(image_path)

# 모드 체크는 convert() 전에 해야 함
if input_img.mode != 'RGBA':
    rembg = BackgroundRemover()
    output_img = rembg(input_img.convert('RGBA'))
else:
    output_img = input_img  # 이미 투명 배경

bg_removed_path = '/kaggle/working/input_no_bg.png'
output_img.save(bg_removed_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(input_img)
axes[0].set_title('원본')
axes[0].axis('off')
axes[1].imshow(output_img)
axes[1].set_title('배경 제거')
axes[1].axis('off')
plt.tight_layout()
plt.show()
print(f'배경 제거 완료: {bg_removed_path}')

In [ ]:
# 6. Hunyuan3D-2.1 Shape  (2-GPU split via accelerate dispatch)
import sys, os, gc
import torch

def gpu_status(label=''):
    if label:
        print('\n[' + label + ']')
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total     = props.total_memory / 1024**3
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved  = torch.cuda.memory_reserved(i) / 1024**3
        print('  GPU ' + str(i) + ': used=' + f'{allocated:.2f}' + 'G'
              + ' | cache=' + f'{reserved-allocated:.2f}' + 'G'
              + ' | free=' + f'{total-reserved:.2f}' + 'G / ' + f'{total:.1f}' + 'G')

def gpu_cleanup(device_id):
    torch.cuda.synchronize(device_id)
    gc.collect()
    torch.cuda.empty_cache()

hy3dshape_root = '/kaggle/working/Hunyuan3D-2.1/hy3dshape'
if hy3dshape_root not in sys.path:
    sys.path.insert(0, hy3dshape_root)

os.makedirs('/kaggle/working/output', exist_ok=True)
gpu_status('before shape')

!pip install accelerate -q
import accelerate as _acc

# tqdm을 notebook 모드로 교체 (긴 한줄 출력 방지)
from tqdm.auto import tqdm as _tqdm_cls
import tqdm as _tqdm_mod
_tqdm_mod.tqdm = _tqdm_cls
try:
    import tqdm.std; tqdm.std.tqdm = _tqdm_cls
except Exception:
    pass

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline

# 1단계: GPU0에 float16으로 로드 (항상 성공)
print('\n[1/3] Loading pipeline on GPU0 float16...')
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1',
    device='cuda:0',
    torch_dtype=torch.float16,
)
gpu_status('after load')

# 2단계: 파이프라인 컴포넌트 파악
print('\n[2/3] Pipeline components:')
_components = []
for _k, _v in vars(pipeline).items():
    if hasattr(_v, 'parameters'):
        _params = list(_v.parameters())
        if _params:
            _gb = sum(p.numel() * p.element_size() for p in _params) / 1e9
            _dev = str(_params[0].device)
            _components.append((_k, _v, _gb))
            print('  ' + _k + ': ' + f'{_gb:.2f}' + 'GB  device=' + _dev)

# 3단계: 가장 큰 컴포넌트를 GPU0+GPU1에 분산
_components.sort(key=lambda x: x[2], reverse=True)
if _components:
    _main_key, _main_mod, _main_gb = _components[0]
    print('\n[3/3] Dispatching ' + _main_key + ' (' + f'{_main_gb:.2f}' + 'GB) -> GPU0+GPU1')
    _main_mod.to('cpu')
    for _i in range(torch.cuda.device_count()):
        gpu_cleanup(_i)
    try:
        _dmap = _acc.infer_auto_device_map(
            _main_mod,
            max_memory={0: '13GiB', 1: '14GiB'},
            dtype=torch.float16,
        )
        setattr(pipeline, _main_key, _acc.dispatch_model(_main_mod, device_map=_dmap))
        _gpu_counts = {}
        for _d in _dmap.values():
            _gpu_counts[str(_d)] = _gpu_counts.get(str(_d), 0) + 1
        print('  Layer distribution: ' + str(_gpu_counts))
    except Exception as _e:
        print('  dispatch failed: ' + str(_e) + '  -> fallback GPU0')
        _main_mod.to('cuda:0')
else:
    print('[3/3] No distributable components found')

gpu_status('after 2-GPU split')

print('\n[shape] Generating 3D...')
with torch.no_grad():
    mesh = pipeline(image=bg_removed_path)[0]

output_path = '/kaggle/working/output/result.glb'
mesh.export(output_path)
print('Saved: ' + output_path)

del mesh, pipeline
for _i in range(torch.cuda.device_count()):
    gpu_cleanup(_i)
gpu_status('after cleanup')


In [ ]:
# 7. (optional) Texture - both GPUs available subprocess
import sys, os, subprocess

gpu_status('before texture')

HY3DPAINT_ROOT = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint'
os.makedirs(f'{HY3DPAINT_ROOT}/ckpt', exist_ok=True)
esrgan_ckpt = f'{HY3DPAINT_ROOT}/ckpt/RealESRGAN_x4plus.pth'

!pip install realesrgan basicsr fast_simplification -q

if not os.path.exists(esrgan_ckpt):
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P {HY3DPAINT_ROOT}/ckpt
    print('RealESRGAN downloaded')

textured_path = '/kaggle/working/output/result_textured.glb'

_worker = """
import os, sys, gc
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
import torch, trimesh
from unittest.mock import MagicMock

print('[worker] GPU count: ' + str(torch.cuda.device_count()))
for _gi in range(torch.cuda.device_count()):
    print('[worker] GPU ' + str(_gi) + ': ' + torch.cuda.get_device_name(_gi))

HY3D = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint'
sys.path = [p for p in sys.path if 'custom_rasterizer' not in p]
sys.path.insert(0, HY3D)
sys.path.insert(0, HY3D + '/custom_rasterizer')

import torchvision.transforms.functional as _tvf
sys.modules['torchvision.transforms.functional_tensor'] = _tvf
sys.modules['bpy'] = MagicMock()

print('[worker] importing custom_rasterizer...')
import custom_rasterizer as _cr
print('[worker] custom_rasterizer exports: ' + str([x for x in dir(_cr) if not x.startswith('_')]))

import utils.simplify_mesh_utils as _smu
import trimesh as _trimesh
def _simplify(src, dst, n=10000):
    mesh = _trimesh.load(src)
    if isinstance(mesh, _trimesh.Scene):
        mesh = _trimesh.util.concatenate(list(mesh.geometry.values()))
    if len(mesh.faces) > n:
        mesh = mesh.simplify_quadric_decimation(face_count=n)
    mesh.export(dst)
    return mesh
_smu.mesh_simplify_trimesh = _simplify
_smu.remesh_mesh = lambda s, d: _simplify(s, d)

import DifferentiableRenderer.mesh_utils as _mu
import DifferentiableRenderer.MeshRender as _mr

# mesh_utils 파일에서 meshVerticeInpaint 정의 확인
import inspect, re
_mu_file = inspect.getfile(_mu)
print('[worker] mesh_utils path: ' + _mu_file)
with open(_mu_file, 'r') as _f:
    _mu_src = _f.read()
for _i, _l in enumerate(_mu_src.splitlines()):
    if 'meshVerticeInpaint' in _l:
        print('[worker] mesh_utils line ' + str(_i) + ': ' + _l.rstrip())

# mesh_utils -> MeshRender 전체 주입
for _fn in dir(_mu):
    if not _fn.startswith('_'):
        setattr(_mr, _fn, getattr(_mu, _fn))

# meshVerticeInpaint callable 확인 및 수정
_mvi = getattr(_mr, 'meshVerticeInpaint', None)
print('[worker] meshVerticeInpaint type: ' + str(type(_mvi).__name__))

if callable(_mvi):
    print('[worker] meshVerticeInpaint callable OK')
else:
    # _OpNamespace인 경우: 네임스페이스 안의 callable 찾기
    _fixed = False

    # 방법 1: torch.ops.meshVerticeInpaint 네임스페이스 직접 확인
    if hasattr(torch.ops, 'meshVerticeInpaint'):
        _ns = torch.ops.meshVerticeInpaint
        _ns_ops = [x for x in dir(_ns) if not x.startswith('_')]
        print('[worker] torch.ops.meshVerticeInpaint contents: ' + str(_ns_ops))
        for _op_nm in _ns_ops:
            _fn = getattr(_ns, _op_nm, None)
            if _fn and callable(_fn):
                _mr.meshVerticeInpaint = _fn
                print('[worker] Fixed: using torch.ops.meshVerticeInpaint.' + _op_nm)
                _fixed = True
                break

    # 방법 2: _mvi 자체가 네임스페이스면 그 안에서 찾기
    if not _fixed and _mvi is not None:
        _ns_ops = [x for x in dir(_mvi) if not x.startswith('_')]
        print('[worker] _mvi namespace contents: ' + str(_ns_ops))
        for _op_nm in _ns_ops:
            _fn = getattr(_mvi, _op_nm, None)
            if _fn and callable(_fn):
                _mr.meshVerticeInpaint = _fn
                print('[worker] Fixed: using _mvi.' + _op_nm)
                _fixed = True
                break

    # 방법 3: custom_rasterizer 전체 검색
    if not _fixed:
        for _attr in [x for x in dir(_cr) if not x.startswith('_')]:
            _fn = getattr(_cr, _attr, None)
            if _fn and callable(_fn) and 'inpaint' in _attr.lower():
                _mr.meshVerticeInpaint = _fn
                print('[worker] Fixed: using _cr.' + _attr)
                _fixed = True
                break

    # 방법 4: OpenCV 기반 fallback 구현
    if not _fixed:
        import numpy as np, cv2
        def _meshVerticeInpaint_cv(texture, mask, vtx_pos, vtx_uv, pos_idx, uv_idx):
            if hasattr(texture, 'cpu'): texture = texture.cpu().numpy()
            if hasattr(mask, 'cpu'):    mask = mask.cpu().numpy()
            tex_u8 = (np.clip(texture, 0, 1) * 255).astype(np.uint8)
            msk_u8 = (mask > 0.5).astype(np.uint8) * 255
            if tex_u8.ndim == 2: tex_u8 = tex_u8[:, :, None]
            result = cv2.inpaint(tex_u8, msk_u8, 3, cv2.INPAINT_TELEA)
            return result.astype(np.float32) / 255.0, mask
        _mr.meshVerticeInpaint = _meshVerticeInpaint_cv
        print('[worker] Fixed: using OpenCV inpaint fallback')
        _fixed = True

gc.collect()
torch.cuda.empty_cache()

from textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
cfg = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
cfg.realesrgan_ckpt_path = HY3D + '/ckpt/RealESRGAN_x4plus.pth'
cfg.multiview_cfg_path   = HY3D + '/cfgs/hunyuan-paint-pbr.yaml'
cfg.device = 'cuda:0'

print('[worker] loading pipeline...')
pipeline = Hunyuan3DPaintPipeline(cfg)
print('[worker] pipeline loaded, starting inference...')

pipeline(
    mesh_path='/kaggle/working/output/result.glb',
    image_path='/kaggle/working/input_no_bg.png',
    output_mesh_path='/kaggle/working/output/result_textured.glb',
    use_remesh=False,
)
del pipeline
gc.collect()
torch.cuda.empty_cache()
print('[worker] texture_done')
"""

_env = os.environ.copy()
_env['PYTHONUNBUFFERED'] = '1'

print('Starting texture subprocess...')
_proc = subprocess.Popen(
    [sys.executable, '-u', '-'],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    env=_env,
)
_proc.stdin.write(_worker)
_proc.stdin.close()

for _line in _proc.stdout:
    # \r 포함 라인은 마지막 청크만 출력 (tqdm 진행바 처리)
    if '\r' in _line:
        _line = _line.split('\r')[-1]
    if _line.strip():
        print(_line, end='\n', flush=True)

_proc.wait()
if _proc.returncode != 0:
    raise RuntimeError(f'Texture subprocess failed (exit code {_proc.returncode})')

gpu_status('after texture')
print(f'Textured: {textured_path}')
print(f'Size: {os.path.getsize(textured_path) / 1024 / 1024:.1f} MB')


In [ ]:
# 8. 결과 확인
import glob, os

output_files = glob.glob('/kaggle/working/output/*.*')
print('생성된 파일:')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

print()
print('Kaggle Output 탭에서 파일을 다운로드하거나,')
print('노트북 저장(Ctrl+S) 후 오른쪽 Output 패널에서 확인하세요.')